<a href="https://colab.research.google.com/github/Saptamabar/TrustChain_AI/blob/main/TrustChain_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files

# Akan memunculkan tombol upload
uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [4]:
# Membuat folder tersembunyi
!mkdir -p ~/.kaggle

# Memindahkan file kaggle.json
!cp kaggle.json ~/.kaggle/

# Mengubah izin akses file demi keamanan
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!pip install --upgrade kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: kagglesdk
    Found existing installation: kagglesdk 0.1.20
    Uninstalling kagglesdk-0.1.20:
      Successfully uninstalled kagglesdk-0.1.20
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


In [6]:
# 1. Unduh tanpa argumen --unzip (akan menghasilkan file .zip)
!kaggle competitions download -c ieee-fraud-detection

# 2. Ekstrak file zip secara manual menggunakan perintah Linux 'unzip'
!unzip ieee-fraud-detection.zip

100% 118M/118M [00:01<00:00, 104MB/s] 

Archive:  ieee-fraud-detection.zip
  inflating: sample_submission.csv   
  inflating: test_identity.csv       
  inflating: test_transaction.csv    
  inflating: train_identity.csv      
  inflating: train_transaction.csv   


In [7]:
import os
import gc  # Garbage Collector untuk membersihkan RAM
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import joblib
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# FUNGSI OPTIMASI MEMORI (Wajib untuk Big Data)
# ─────────────────────────────────────────────
def reduce_mem_usage(df):
    """Iterasi semua kolom dataframe dan ubah tipe data untuk menghemat memori"""
    start_mem = df.memory_usage().sum() / 1024**2
    logger.info(f'Memori awal: {start_mem:.2f} MB')

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    logger.info(f'Memori akhir setelah optimasi: {end_mem:.2f} MB')
    return df

# ─────────────────────────────────────────────
# 1. LOAD DATA & OPTIMASI
# ─────────────────────────────────────────────
def load_data(data_dir: str):
    logger.info("Loading transaction & identity data...")

    train_txn = pd.read_csv(os.path.join(data_dir, "train_transaction.csv"))
    train_id  = pd.read_csv(os.path.join(data_dir, "train_identity.csv"))
    test_txn  = pd.read_csv(os.path.join(data_dir, "test_transaction.csv"))
    test_id   = pd.read_csv(os.path.join(data_dir, "test_identity.csv"))

    # Seragamkan nama kolom test_id
    test_id.columns = test_id.columns.str.replace('-', '_')

    # Merge data
    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id,  on="TransactionID", how="left")

    # Hapus dataframe mentah dari memori karena sudah digabung
    del train_txn, train_id, test_txn, test_id
    gc.collect() # Panggil tukang sampah memori

    # Terapkan optimasi downcasting
    logger.info("Mengecilkan ukuran memori Train...")
    train = reduce_mem_usage(train)
    logger.info("Mengecilkan ukuran memori Test...")
    test = reduce_mem_usage(test)

    logger.info(f"Train shape: {train.shape}, Test shape: {test.shape}")
    return train, test

# ─────────────────────────────────────────────
# 2. FEATURE ENGINEERING DASAR
# ─────────────────────────────────────────────
def feature_engineering(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    # Hapus .copy() agar tidak menggandakan pemakaian RAM
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

    df["hour"] = (df["TransactionDT"] // 3600) % 24
    df["day"]  = (df["TransactionDT"] // 86400) % 7

    transaction_day = df["TransactionDT"] // 86400
    d_cols = [f"D{i}" for i in range(1, 16) if f"D{i}" in df.columns]

    for col in d_cols:
        df[f"{col}_norm"] = df[col] - transaction_day

    df.drop(columns=d_cols, inplace=True)
    gc.collect() # Bersihkan RAM lagi
    return df

# ─────────────────────────────────────────────
# 3. CLEANING: MISSING VALUES & ENCODING
# ─────────────────────────────────────────────
def clean_data(train: pd.DataFrame, test: pd.DataFrame,
               missing_threshold: float = 0.9,
               output_dir: str = "/content/artifacts"):
    os.makedirs(output_dir, exist_ok=True)

    label_col = "isFraud"
    id_col    = "TransactionID"

    y_train = train[label_col].values if label_col in train.columns else None

    train.drop(columns=[id_col, label_col], inplace=True, errors="ignore")
    test.drop(columns=[id_col], inplace=True, errors="ignore")

    missing_rate = train.isnull().mean()
    drop_cols = missing_rate[missing_rate > missing_threshold].index.tolist()
    train.drop(columns=drop_cols, inplace=True, errors="ignore")
    test.drop(columns=drop_cols,  inplace=True, errors="ignore")
    gc.collect()

    cat_cols = train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = train.select_dtypes(exclude=["object", "category"]).columns.tolist()

    train[cat_cols] = train[cat_cols].fillna("missing")
    test[cat_cols]  = test[cat_cols].fillna("missing")

    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        # Fit terpisah untuk menghemat memori pembuatan variabel combined
        le.fit(list(train[col].astype(str).values) + list(test[col].astype(str).values))
        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))
        encoders[col] = le

    joblib.dump(encoders, os.path.join(output_dir, "label_encoders.pkl"))

    num_imputer = SimpleImputer(strategy="median")
    train[num_cols] = num_imputer.fit_transform(train[num_cols])
    test[num_cols]  = num_imputer.transform(test[num_cols])
    joblib.dump(num_imputer, os.path.join(output_dir, "num_imputer.pkl"))

    # Simpan nama kolom
    feature_names = train.columns.tolist()
    joblib.dump(feature_names, os.path.join(output_dir, "feature_names.pkl"))

    # Convert ke Float32 sebelum scaling agar RAM tidak meledak di StandardScaler
    train = train.astype(np.float32)
    test = test.astype(np.float32)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train)
    test_scaled  = scaler.transform(test)
    joblib.dump(scaler, os.path.join(output_dir, "scaler.pkl"))

    # Hapus dataframe pandas untuk membebaskan memori sebelum save Numpy
    del train, test
    gc.collect()

    np.save(os.path.join(output_dir, "X_train.npy"), train_scaled)
    np.save(os.path.join(output_dir, "X_test.npy"),  test_scaled)
    if y_train is not None:
        np.save(os.path.join(output_dir, "y_train.npy"), y_train)

    logger.info(f"Data bersih disimpan di: {output_dir}")
    return True

# ─────────────────────────────────────────────
# EKSEKUSI DI COLAB
# ─────────────────────────────────────────────
DATA_DIR = "/content/"
OUTPUT_DIR = "/content/artifacts"

print("Mulai proses persiapan data...")
df_train, df_test = load_data(DATA_DIR)

df_train = feature_engineering(df_train, is_train=True)
df_test  = feature_engineering(df_test,  is_train=False)

clean_data(df_train, df_test, missing_threshold=0.9, output_dir=OUTPUT_DIR)

print("\n✅ Selesai! Semua artefak siap.")

Mulai proses persiapan data...


/tmp/ipykernel_3816/404020493.py:37: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_3816/404020493.py:37: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_3816/404020493.py:37: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_3816/404020493.py:37: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_3816/404020493.py:37: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_3816/404020493.py:37: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_3816/404020493.py:37: RuntimeWarning:


✅ Selesai! Semua artefak siap.


In [8]:
import os
import gc
import numpy as np
import joblib
import logging
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix)

# Konfigurasi Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# 1. TRAIN ISOLATION FOREST
# ─────────────────────────────────────────────
def train_isolation_forest(X_train: np.ndarray,
                           contamination: float = 0.035,
                           n_estimators: int = 200,
                           random_state: int = 42) -> IsolationForest:
    logger.info(f"Training Isolation Forest | contamination={contamination} | n_estimators={n_estimators}")

    model = IsolationForest(
        n_estimators=n_estimators,
        contamination=contamination,
        max_samples="auto",
        max_features=1.0,
        bootstrap=False,
        n_jobs=-1, # Gunakan semua core CPU yang tersedia di Colab
        random_state=random_state,
        verbose=0,
    )
    model.fit(X_train)
    logger.info("Isolation Forest selesai dilatih.")
    return model

# ─────────────────────────────────────────────
# 2. GENERATE ANOMALY SCORES
# ─────────────────────────────────────────────
def get_anomaly_scores(model: IsolationForest, X: np.ndarray) -> np.ndarray:
    # decision_function mengembalikan nilai negatif untuk anomali
    raw_scores = model.decision_function(X)

    # Invert & normalisasi ke [0,1] agar model LSTM lebih mudah memprosesnya nanti
    scores_inv = -raw_scores
    scores_norm = (scores_inv - scores_inv.min()) / (scores_inv.max() - scores_inv.min() + 1e-9)
    return scores_norm

# ─────────────────────────────────────────────
# 3. EVALUASI
# ─────────────────────────────────────────────
def evaluate(model: IsolationForest, X: np.ndarray, y_true: np.ndarray):
    preds_raw  = model.predict(X)                    # Output bawaan: 1 (normal), -1 (anomali)
    y_pred     = np.where(preds_raw == -1, 1, 0)     # Konversi ke target kita: 1 (fraud), 0 (normal)

    scores_norm = get_anomaly_scores(model, X)

    auc = roc_auc_score(y_true, scores_norm)
    logger.info(f"ROC-AUC Isolation Forest: {auc:.4f}")
    logger.info("\nClassification Report:\n" +
                classification_report(y_true, y_pred, target_names=["Normal","Fraud"]))

    cm = confusion_matrix(y_true, y_pred)
    logger.info(f"Confusion Matrix:\n{cm}")
    return auc

# ─────────────────────────────────────────────
# 4. SIMPAN ARTEFAK
# ─────────────────────────────────────────────
def save_artifacts(model, X_train, X_test, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # Simpan model
    model_path = os.path.join(output_dir, "isolation_forest.pkl")
    joblib.dump(model, model_path)
    logger.info(f"Model IF disimpan → {model_path}")

    # Hitung anomaly scores
    logger.info("Menghitung Anomaly Score untuk Train set...")
    train_scores = get_anomaly_scores(model, X_train)

    logger.info("Menghitung Anomaly Score untuk Test set...")
    test_scores  = get_anomaly_scores(model, X_test)

    # Simpan skor murni (opsional)
    np.save(os.path.join(output_dir, "if_scores_train.npy"), train_scores)
    np.save(os.path.join(output_dir, "if_scores_test.npy"),  test_scores)

    # Augmentasi: Gabungkan fitur lama dengan skor anomali sebagai fitur baru
    logger.info("Menggabungkan skor anomali ke dalam matriks fitur utama...")
    X_train_aug = np.hstack([X_train, train_scores.reshape(-1, 1)])
    X_test_aug  = np.hstack([X_test,  test_scores.reshape(-1, 1)])

    # Simpan matriks fitur yang sudah diperkaya
    np.save(os.path.join(output_dir, "X_train_aug.npy"), X_train_aug)
    np.save(os.path.join(output_dir, "X_test_aug.npy"),  X_test_aug)

    logger.info(f"X_train_aug shape: {X_train_aug.shape}, X_test_aug shape: {X_test_aug.shape}")

    return X_train_aug, X_test_aug


# ─────────────────────────────────────────────
# EKSEKUSI DI COLAB
# ─────────────────────────────────────────────
ARTIFACTS_DIR = "/content/artifacts"
CONTAMINATION = 0.035
N_ESTIMATORS = 200

print("Mulai memuat data Numpy...")
# Load data hasil cleaning dari step 1
X_train = np.load(os.path.join(ARTIFACTS_DIR, "X_train.npy"))
X_test  = np.load(os.path.join(ARTIFACTS_DIR, "X_test.npy"))

y_path = os.path.join(ARTIFACTS_DIR, "y_train.npy")
y_train = np.load(y_path) if os.path.exists(y_path) else None

# Train Model
model_if = train_isolation_forest(
    X_train,
    contamination=CONTAMINATION,
    n_estimators=N_ESTIMATORS,
)

# Evaluasi jika label tersedia
if y_train is not None:
    print("\nMelakukan evaluasi model...")
    evaluate(model_if, X_train, y_train)

# Simpan model dan buat matriks fitur baru
print("\nMenyimpan artefak...")
save_artifacts(model_if, X_train, X_test, ARTIFACTS_DIR)

# Bersihkan RAM setelah selesai
del X_train, X_test, model_if
gc.collect()

print("\n✅ Step 2 Selesai! Fitur tambahan dari Isolation Forest berhasil dibuat.")

Mulai memuat data Numpy...

Melakukan evaluasi model...

Menyimpan artefak...

✅ Step 2 Selesai! Fitur tambahan dari Isolation Forest berhasil dibuat.


In [9]:
import os
import numpy as np
import logging
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, backend as K
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Konfigurasi Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# 1. FOCAL LOSS (lebih baik untuk imbalanced)
# ─────────────────────────────────────────────
def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """
    Focal Loss untuk menangani class imbalance.
    gamma=2 -> fokus pada contoh sulit
    alpha=0.25 -> bobot kelas positif (fraud)
    """
    def focal_loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce    = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t    = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        weight = tf.where(tf.equal(y_true, 1),
                          tf.ones_like(y_true) * alpha,
                          tf.ones_like(y_true) * (1 - alpha))
        focal  = weight * tf.pow(1 - p_t, gamma) * bce
        return tf.reduce_mean(focal)
    return focal_loss_fn

# ─────────────────────────────────────────────
# 2. BANGUN MODEL LSTM
# ─────────────────────────────────────────────
def build_lstm_model(n_features: int,
                     lstm_units: list = [128, 64],
                     dense_units: list = [64, 32],
                     dropout_rate: float = 0.3,
                     learning_rate: float = 1e-3) -> keras.Model:
    inputs = keras.Input(shape=(1, n_features), name="transaction_input")
    x = inputs

    # LSTM layers
    for i, units in enumerate(lstm_units):
        return_seq = (i < len(lstm_units) - 1)   # True kecuali layer terakhir
        x = layers.LSTM(
            units,
            return_sequences=return_seq,
            dropout=dropout_rate,
            recurrent_dropout=0.0, # Ubah ke 0.0 di Colab agar bisa pakai cuDNN GPU (lebih cepat)
            name=f"lstm_{i+1}"
        )(x)
        x = layers.BatchNormalization(name=f"bn_lstm_{i+1}")(x)

    # Dense layers
    for i, units in enumerate(dense_units):
        x = layers.Dense(units, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"bn_dense_{i+1}")(x)
        x = layers.Dropout(dropout_rate, name=f"dropout_{i+1}")(x)

    # Output
    output = layers.Dense(1, activation="sigmoid", name="fraud_prob")(x)

    model = keras.Model(inputs=inputs, outputs=output, name="FraudLSTM")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=focal_loss(gamma=2.0, alpha=0.25),
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ]
    )
    return model

# ─────────────────────────────────────────────
# 3. TRAINING
# ─────────────────────────────────────────────
def train_lstm(X_train: np.ndarray, y_train: np.ndarray, output_dir: str,
               epochs: int = 30, batch_size: int = 2048,
               val_split: float = 0.15, learning_rate: float = 1e-3):
    os.makedirs(output_dir, exist_ok=True)

    n_samples, n_features = X_train.shape
    logger.info(f"Shape input: {X_train.shape}, Labels: {y_train.shape}")

    # Reshape -> (samples, timesteps=1, features)
    X_3d = X_train.reshape(n_samples, 1, n_features)

    # Class weight
    classes = np.unique(y_train)
    cw_vals = compute_class_weight("balanced", classes=classes, y=y_train)
    class_weight = {int(c): float(w) for c, w in zip(classes, cw_vals)}
    logger.info(f"Class weights: {class_weight}")

    # Build model
    model = build_lstm_model(n_features=n_features, learning_rate=learning_rate)
    model.summary(print_fn=logger.info)

    # Callbacks
    cb_list = [
        callbacks.EarlyStopping(monitor="val_auc", patience=5, mode="max", restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_auc", factor=0.5, patience=3, mode="max", min_lr=1e-6, verbose=1),
        callbacks.ModelCheckpoint(filepath=os.path.join(output_dir, "best_lstm.keras"), monitor="val_auc", save_best_only=True, mode="max", verbose=1),
        callbacks.CSVLogger(os.path.join(output_dir, "training_log.csv"), append=False)
    ]

    # Training
    logger.info(f"Mulai training: {epochs} epochs, batch={batch_size}")
    history = model.fit(
        X_3d, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=val_split,
        class_weight=class_weight,
        callbacks=cb_list,
        verbose=1,
    )
    logger.info("Training selesai.")
    return model, history

# ─────────────────────────────────────────────
# 4. EVALUASI
# ─────────────────────────────────────────────
def evaluate_lstm(model: keras.Model, X: np.ndarray, y_true: np.ndarray, threshold: float = 0.5):
    n_samples, n_features = X.shape
    X_3d = X.reshape(n_samples, 1, n_features)

    y_prob = model.predict(X_3d, batch_size=2048).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_prob)
    logger.info(f"\n{'='*40}")
    logger.info(f"ROC-AUC LSTM: {auc:.4f}")
    logger.info("\nClassification Report:\n" + classification_report(y_true, y_pred, target_names=["Normal","Fraud"]))
    logger.info(f"{'='*40}")
    return y_prob, auc

# ─────────────────────────────────────────────
# 5. PREDIKSI & SIMPAN SUBMISSION
# ─────────────────────────────────────────────
def generate_submission(model: keras.Model, X_test: np.ndarray, output_dir: str):
    import pandas as pd
    n_samples, n_features = X_test.shape
    X_3d = X_test.reshape(n_samples, 1, n_features)

    y_prob = model.predict(X_3d, batch_size=2048).ravel()

    # Membuat template submission dasar
    submission = pd.DataFrame({
        "TransactionID": np.arange(len(y_prob)), # Ini hanya placeholder. Di dunia nyata, gabungkan dengan TransactionID dari test_transaction.csv
        "isFraud": y_prob
    })
    sub_path = os.path.join(output_dir, "submission.csv")
    submission.to_csv(sub_path, index=False)
    logger.info(f"Submission disimpan -> {sub_path}")
    return y_prob


# ─────────────────────────────────────────────
# EKSEKUSI DI COLAB
# ─────────────────────────────────────────────
ARTIFACTS_DIR = "/content/artifacts"
OUTPUT_DIR = "/content/artifacts"
EPOCHS = 30
BATCH_SIZE = 2048  # Cocok untuk memori GPU Colab
VAL_SPLIT = 0.15
LR = 1e-3
USE_AUGMENTED = True # Gunakan True agar memuat X_train_aug.npy dari Isolation Forest

print("Mempersiapkan data...")
if USE_AUGMENTED:
    logger.info("Menggunakan dataset yang sudah disematkan fitur Anomaly Score dari Isolation Forest...")
    X_train = np.load(os.path.join(ARTIFACTS_DIR, "X_train_aug.npy"))
    X_test  = np.load(os.path.join(ARTIFACTS_DIR, "X_test_aug.npy"))
else:
    logger.info("Menggunakan dataset standar...")
    X_train = np.load(os.path.join(ARTIFACTS_DIR, "X_train.npy"))
    X_test  = np.load(os.path.join(ARTIFACTS_DIR, "X_test.npy"))

y_train = np.load(os.path.join(ARTIFACTS_DIR, "y_train.npy"))

print("\n🚀 Memulai Proses Training...")
model, history = train_lstm(
    X_train, y_train,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    learning_rate=LR,
)

print("\n📊 Mengevaluasi Model...")
evaluate_lstm(model, X_train, y_train)

print("\n💾 Menyimpan File Prediksi (Submission)...")
generate_submission(model, X_test, OUTPUT_DIR)

print("\n✅ Step 3 Selesai! Model LSTM terbaik kamu ('best_lstm.keras') telah tersimpan di folder artifacts.")

Mempersiapkan data...

🚀 Memulai Proses Training...


Epoch 1/30
244/246 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - auc: 0.5085 - loss: 0.1755 - precision: 0.0405 - recall: 0.3489
Epoch 1: val_auc improved from None to 0.63088, saving model to /content/artifacts/best_lstm.keras

Epoch 1: finished saving model to /content/artifacts/best_lstm.keras
246/246 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - auc: 0.5098 - loss: 0.0995 - precision: 0.0455 - recall: 0.2092 - val_auc: 0.6309 - val_loss: 0.0217 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/30
246/246 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - auc: 0.6221 - loss: 0.0219 - precision: 0.2789 - recall: 0.0449
Epoch 2: val_auc improved from 0.63088 to 0.78884, saving model to /content/artifacts/best_lstm.keras

Epoch 2: finished saving model to /content/artifacts/best_lstm.keras
246/246 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - auc: 0.6521 - loss: 0.0181 - precision: 0.4096 - recall: 0.0509 - val_auc: 0.7888 - val_loss: 0.0122 - val_precision: 0.9208 - val_recall: 0.0717 - learn